# A Compliance Agent in One Notebook <sup>🔐 NIST CM-7</sup>

**The task:** you have a hardening standard written in English. Prove this machine complies with it, and produce the evidence.

Today that is a person with a checklist and a terminal. Here we give a language model two tools and let it do the run.

The standard is real. **NIST SP 800-53 Rev 5, control CM-7 "Least Functionality"** says, verbatim:

> Prohibit or restrict the use of the following functions, ports, protocols, software, and/or services: *[organisation-defined]*.

Note the last two words — we come back to them, because they are the whole point.

**Scope, deliberately narrow and safe:** we check TCP ports on **this notebook's own machine, `127.0.0.1` only**. The tool refuses any other host. It reports; it never changes anything. Scanning a machine you do not own is a different activity with a different set of consequences, and nothing here should be pointed off-box.

In [1]:
#@title setup
import json, re, socket, torch, pandas as pd
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


## 1. The policy, in English

Five lines of a hardening standard. This is the input a compliance engineer is actually handed — prose, not configuration.

In [2]:
#@title the hardening standard
POLICY = """
1. SSH (port 22) must be reachable for administration.
2. Telnet (port 23) is obsolete and must be disabled.
3. Unencrypted HTTP (port 80) must not be served from this host.
4. SMB file sharing (port 445) must be disabled.
5. Remote desktop (port 3389) must be disabled.
"""
print(POLICY)


1. SSH (port 22) must be reachable for administration.
2. Telnet (port 23) is obsolete and must be disabled.
3. Unencrypted HTTP (port 80) must not be served from this host.
4. SMB file sharing (port 445) must be disabled.
5. Remote desktop (port 3389) must be disabled.



## 2. The tools

Two functions, and the safety rail is in the code rather than in the prompt. `check_port` will not accept a remote host — a model cannot talk it into scanning something else, because refusing is not a decision the model gets to make.

In [3]:
#@title tools the agent may call
LOCAL = {"127.0.0.1", "localhost", "::1"}

def check_port(port: int, expected: str, host: str = "127.0.0.1") -> dict:
    """Is `port` open or closed on THIS machine? expected is 'open' or 'closed'."""
    if host not in LOCAL:
        raise PermissionError(f"refusing to probe {host!r}: this tool only inspects the local machine")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.4)
        observed = "open" if s.connect_ex(("127.0.0.1", int(port))) == 0 else "closed"
    return {"port": int(port), "expected": expected, "observed": observed,
            "compliant": observed == expected}

def list_listening_ports(scan_to: int = 1024) -> list:
    """Which low TCP ports are actually listening here?"""
    out = []
    for p in range(1, scan_to + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.05)
            if s.connect_ex(("127.0.0.1", p)) == 0: out.append(p)
    return out

print("listening below 1024:", list_listening_ports())
print("smoke test:", check_port(22, "open"))
try:
    check_port(22, "open", host="scanme.nmap.org")
except PermissionError as e:
    print("guard works:", e)

listening below 1024: [22, 53]
smoke test: {'port': 22, 'expected': 'open', 'observed': 'open', 'compliant': True}
guard works: refusing to probe 'scanme.nmap.org': this tool only inspects the local machine


## 3. The agent

The model's job is narrow and checkable: **read the prose, emit the checks**. It does not decide policy and it does not touch the machine — it produces a list of `{port, expected}` pairs, and our code runs them.

That split is the design point. The model works in language, where it is strong; the execution stays in code, where it is auditable.

In [4]:
#@title load a small instruct model
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct" if device == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
print("model:", MODEL_ID)

/Users/mohammadalshiekh/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


model: Qwen/Qwen2.5-0.5B-Instruct


In [5]:
#@title prose -> checks
SYSTEM = ("You convert a security policy into port checks. "
          "Reply with JSON only: a list of objects with keys \"port\" (integer) "
          "and \"expected\" (either \"open\" or \"closed\"). No prose, no code fences.")

@torch.no_grad()
def plan_checks(policy_text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": policy_text.strip()}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt").to(device)
    out = llm.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.eos_token_id)
    txt = tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"\[.*\]", txt, re.S)
    if not m: raise ValueError(f"no JSON list in model output:\n{txt}")
    return json.loads(m.group(0)), txt

checks, raw = plan_checks(POLICY)
print("the agent proposed these checks:\n")
for c in checks: print(" ", c)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


the agent proposed these checks:

  {'port': 22, 'expected': 'open'}
  {'port': 23, 'expected': 'closed'}
  {'port': 80, 'expected': 'closed'}
  {'port': 445, 'expected': 'closed'}
  {'port': 3389, 'expected': 'closed'}


In [6]:
#@title run them and produce the evidence
results = [check_port(c["port"], c["expected"]) for c in checks]
report = pd.DataFrame(results)
report["verdict"] = report.compliant.map({True: "PASS", False: "FAIL"})
print("NIST SP 800-53 CM-7 (Least Functionality) - evidence for this host\n")
print(report[["port", "expected", "observed", "verdict"]].to_string(index=False))
n_fail = int((~report.compliant).sum())
print(f"\n{len(report) - n_fail}/{len(report)} checks pass"
      + ("" if n_fail == 0 else f"  -  {n_fail} finding(s) to remediate"))

NIST SP 800-53 CM-7 (Least Functionality) - evidence for this host



 port expected observed verdict
   22     open     open    PASS
   23   closed   closed    PASS
   80   closed   closed    PASS
  445   closed   closed    PASS
 3389   closed   closed    PASS

5/5 checks pass


## 4. What this does and does not prove

**What just happened.** The model read five English sentences and produced an executable check list; our code ran it against the local host and wrote an evidence table. That table is the audit deliverable — a finding without the raw observation behind it is an opinion.

**Now the honest part.** For five known ports, a hand-written checklist does the same job, never hallucinates, and needs no GPU. The model is not beating the checklist here and it is not meant to. Where it starts to pay is the long tail: a standard with hundreds of clauses, written in prose, that nobody has yet turned into checks — drafting those is the tedious part, and drafting is what it is good at.

**And the trap, which is in the standard itself.** CM-7 does not say which ports to prohibit. It says *organisation-defined*. NIST deliberately leaves that blank because only you know what this machine is for. So if you hand the model the control text alone and ask "are we compliant?", anything it returns is invented — it has no way to know your baseline. **A human supplies the policy; the model translates it; the code enforces it.** Any tool that claims to skip the first step is selling compliance theatre.

**Worth trying next.** Feed it a policy line it cannot satisfy with these two tools ("audit logging must be enabled") and watch what it does — a good agent should decline, a weak one invents a port number. That failure mode is the one to show a manager.